# 04 — Address & SIC Exploration
Cluster companies by shared registered address and SIC activity codes.

In [1]:
import sys
sys.path.insert(0, "..")

In [2]:
import pandas as pd
from src.config import get_neo4j_settings
from src.storage.neo4j_repository import Neo4jRepository

settings = get_neo4j_settings()
repo = Neo4jRepository(**vars(settings))
repo.test_connection()
print("Connected")

Connected


In [3]:
COMPANY_NAME = "TELEFONICA O2 HOLDINGS LIMITED"  # replace as needed

## Registered address

In [4]:
address = repo.get_company_address_context(COMPANY_NAME)

if address:
    display(pd.DataFrame([address]))
else:
    print("No address found — check COMPANY_NAME or schema.")

Received notification from DBMS server: <GqlStatusObject gql_status='01N52', status_description='warn: property key does not exist. The property `locality` does not exist in database `neo4j`. Verify that the spelling is correct.', position=<SummaryInputPosition line=6, column=19, offset=222>, raw_classification='UNRECOGNIZED', classification=<NotificationClassification.UNRECOGNIZED: 'UNRECOGNIZED'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'UNRECOGNIZED', '_severity': 'WARNING', '_position': {'offset': 222, 'line': 6, 'column': 19}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: '\n            MATCH (c:Company {name: $name})-[:REGISTERED_AT]->(a:Address)\n            RETURN\n                a.address_line_1    AS address_line_1,\n                a.address_line_2    AS address_line_2,\n                a.locality          AS locality,\n                a.region            AS region,\n     

,address_line_1,address_line_2,locality,region,postal_code,country
0,HIGHDOWN HOUSE,YEOMAN WAY,None,None,None,UNITED KINGDOM


## Companies at the same address
A shared address node may indicate a registered agent, shell cluster, or group HQ.

In [5]:
co_located = repo.get_companies_at_same_address(COMPANY_NAME, limit=50)
print(f"Companies sharing the same address: {len(co_located)}")

if co_located:
    df_addr = pd.DataFrame(co_located)
    display(df_addr)
else:
    print("None — address is unique to this company.")

Companies sharing the same address: 50


,company_name,company_number,status,postal_code,address_line_1
0,00084968 LIMITED,00084968,Active,None,HIGHDOWN HOUSE
1,15 SPITAL SQUARE RESIDENTS ASSOCIATION LTD,02197789,Active,None,HIGHDOWN HOUSE
2,"AKARI THERAPEUTICS, PLC",05252842,Active,None,HIGHDOWN HOUSE
3,ALPHAWAVE IP GROUP PLC,13073661,Active,None,HIGHDOWN HOUSE
4,ARCUS INVESTMENT HOLDINGS LIMITED,12974482,Active,None,HIGHDOWN HOUSE
5,ARCUS INVESTMENT LIMITED,03582673,Active,None,HIGHDOWN HOUSE
6,ASA INTERNATIONAL GROUP PLC,11361159,Active,None,HIGHDOWN HOUSE
7,BALTIC CLASSIFIEDS GROUP PLC,13357598,Active,None,HIGHDOWN HOUSE
8,BENCHMARK ANIMAL HEALTH GROUP LIMITED,07330728,Active,None,HIGHDOWN HOUSE
9,BENCHMARK ANIMAL HEALTH LIMITED,08872045,Active,None,HIGHDOWN HOUSE


### Address clustering signal
High co-location counts at a single postal code warrant further investigation.

In [6]:
if co_located:
    total = len(co_located)
    active = sum(1 for r in co_located if r.get("status") == "Active")
    print(f"Total co-located  : {total}")
    print(f"Active companies  : {active}")
    print(f"Dissolved/other   : {total - active}")

Total co-located  : 50
Active companies  : 48
Dissolved/other   : 2


## SIC codes for target company

In [7]:
sics = repo.get_company_sic_context(COMPANY_NAME)
print(f"SIC codes: {len(sics)}")

if sics:
    display(pd.DataFrame(sics))
else:
    print("No SIC codes found.")

SIC codes: 2


,sic_code,sic_description
0,None,Other telecommunications activities
1,None,Non-trading company


## Companies sharing the same SIC
Ordered by number of shared codes descending — highest overlap first.

In [8]:
same_sic = repo.get_companies_with_same_sic(COMPANY_NAME, limit=50)
print(f"Companies sharing at least one SIC: {len(same_sic)}")

if same_sic:
    df_sic = pd.DataFrame(same_sic)
    display(df_sic)
else:
    print("None found.")

Companies sharing at least one SIC: 50


,company_name,company_number,status,shared_sic_codes,shared_sic_descriptions
0,"""""""BEECHBANK COURT"""" MANAGEMENT COMPANY LIMITED""",01382560,Active,[],[Non-trading company]
1,"""""""CANNON CHAMBERS"""" 17 CANNON PLACE MANAGEMEN...",02076454,Active,[],[Non-trading company]
2,"""""""INTOUCH COMMUNICATION SERVICES"""" LIMITED""",03606467,Active,[],[Other telecommunications activities]
3,"""""""ST MARY'S HEATH"""" AT ARMTHORPE PROPERTY MAN...",05424123,Active,[],[Non-trading company]
4,"""CHADWELL HEATH """"A"""" FLAT MANAGEMENT LIMITED""",02184970,Active,[],[Non-trading company]
5,"""CHADWELL HEATH """"B"""" FLAT MANAGEMENT LIMITED""",02185149,Active,[],[Non-trading company]
6,"""CHADWELL HEATH """"B"""" MANAGEMENT LIMITED""",01823617,Active,[],[Non-trading company]
7,"""CHADWELL HEATH """"C"""" MANAGEMENT LIMITED""",01823618,Active,[],[Non-trading company]
8,"""CHADWELL HEATH """"D"""" MANAGEMENT LIMITED""",01941575,Active,[],[Non-trading company]
9,"""CHADWELL HEATH """"E"""" MANAGEMENT LIMITED""",01950602,Active,[],[Non-trading company]


### SIC overlap breakdown
How many companies share all SIC codes vs. only a subset.

In [9]:
if same_sic and sics:
    total_sics = len(sics)
    df_sic = pd.DataFrame(same_sic)
    df_sic["shared_count"] = df_sic["shared_sic_codes"].apply(len)
    df_sic["full_overlap"] = df_sic["shared_count"] == total_sics
    summary = df_sic.groupby("shared_count").size().reset_index(name="companies")
    summary["label"] = summary["shared_count"].apply(
        lambda n: f"{n}/{total_sics} codes shared"
    )
    display(summary[["label", "companies"]])

,label,companies
0,0/2 codes shared,50


## Close

In [10]:
repo.close()
print("Driver closed")

Driver closed
